# 05 - Cohorts and Time Series Deep Dive

This notebook performs cohort retention and daily demand time-series diagnostics.


In [ ]:
# Cell 0: Setup and source extraction
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sqlalchemy import inspect, text

from ml.pipelines.churn_common import create_db_engine

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent.parent

OUT_DIR = ROOT / 'ml' / 'data' / 'reports' / 'causal' / 'cohort_time_series'
OUT_DIR.mkdir(parents=True, exist_ok=True)

engine = create_db_engine()
inspector = inspect(engine)
orders_cols = {c['name'] for c in inspector.get_columns('orders')}
amount_col = 'total_amount' if 'total_amount' in orders_cols else 'total_price'

orders = pd.read_sql(
    text(f"SELECT user_id::text as user_id, created_at, {amount_col}::double precision as order_amount FROM orders"),
    engine,
)
orders['created_at'] = pd.to_datetime(orders['created_at'], utc=True, errors='coerce')
orders['order_amount'] = pd.to_numeric(orders['order_amount'], errors='coerce').fillna(0.0)
orders = orders.dropna(subset=['created_at'])
print('orders shape:', orders.shape)


In [ ]:
# Cell 1: Cohort retention table
orders['order_month'] = orders['created_at'].dt.to_period('M').dt.to_timestamp()
first = orders.groupby('user_id', as_index=False)['order_month'].min().rename(columns={'order_month': 'cohort_month'})
cohort_df = orders.merge(first, on='user_id', how='left')
cohort_df['period'] = (
    (cohort_df['order_month'].dt.year - cohort_df['cohort_month'].dt.year) * 12
    + (cohort_df['order_month'].dt.month - cohort_df['cohort_month'].dt.month)
).astype(int)

cohort_metrics = cohort_df.groupby(['cohort_month', 'period'], as_index=False).agg(
    active_users=('user_id', 'nunique'),
    orders=('user_id', 'size'),
    revenue=('order_amount', 'sum'),
)

size = cohort_metrics[cohort_metrics['period'] == 0][['cohort_month', 'active_users', 'revenue']].rename(
    columns={'active_users': 'cohort_size', 'revenue': 'cohort_rev0'}
)
cohort_metrics = cohort_metrics.merge(size, on='cohort_month', how='left')
cohort_metrics['retention_rate'] = (cohort_metrics['active_users'] / cohort_metrics['cohort_size'].replace(0, np.nan)).fillna(0.0)
cohort_metrics['revenue_retention_rate'] = (cohort_metrics['revenue'] / cohort_metrics['cohort_rev0'].replace(0, np.nan)).fillna(0.0)

display(cohort_metrics.head(20))


In [ ]:
# Cell 2: Cohort retention heatmap
pivot = cohort_metrics.pivot(index='cohort_month', columns='period', values='retention_rate').fillna(0.0)

fig, ax = plt.subplots(figsize=(10, 5))
h = ax.imshow(pivot.values, aspect='auto', cmap='YlGnBu', vmin=0, vmax=1)
ax.set_title('Cohort Retention Heatmap')
ax.set_xlabel('Period')
ax.set_ylabel('Cohort Month')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([pd.Timestamp(x).strftime('%Y-%m') for x in pivot.index])
fig.colorbar(h, ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 3: Daily time series and anomalies
orders['order_day'] = orders['created_at'].dt.floor('D')
daily = orders.groupby('order_day', as_index=False).agg(
    orders=('user_id', 'size'),
    revenue=('order_amount', 'sum'),
    users=('user_id', 'nunique'),
).sort_values('order_day')

full_days = pd.date_range(daily['order_day'].min(), daily['order_day'].max(), freq='D', tz='UTC')
daily = daily.set_index('order_day').reindex(full_days, fill_value=0).reset_index().rename(columns={'index': 'order_day'})

daily['orders_7d_ma'] = daily['orders'].rolling(7, min_periods=2).mean()
daily['orders_28d_ma'] = daily['orders'].rolling(28, min_periods=7).mean()
daily['orders_28d_std'] = daily['orders'].rolling(28, min_periods=7).std().fillna(0.0)
daily['orders_zscore'] = ((daily['orders'] - daily['orders_28d_ma']) / daily['orders_28d_std'].replace(0, np.nan)).fillna(0.0)
daily['is_order_anomaly'] = daily['orders_zscore'].abs() >= 2.5

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily['order_day'], daily['orders'], alpha=0.35, label='orders')
ax.plot(daily['order_day'], daily['orders_7d_ma'], label='7d_ma')
ax.plot(daily['order_day'], daily['orders_28d_ma'], label='28d_ma')
pts = daily[daily['is_order_anomaly']]
ax.scatter(pts['order_day'], pts['orders'], color='red', s=16, label='anomaly')
ax.legend(); ax.grid(alpha=0.2); ax.set_title('Daily Orders + Anomalies')
plt.show()


In [ ]:
# Cell 4: Save outputs
cohort_metrics.to_csv(OUT_DIR / 'cohort_metrics.csv', index=False)
pivot.to_csv(OUT_DIR / 'cohort_retention_matrix.csv')
daily.to_csv(OUT_DIR / 'daily_timeseries_metrics.csv', index=False)

summary = {
    'cohorts': int(cohort_metrics['cohort_month'].nunique()),
    'max_period': int(cohort_metrics['period'].max()),
    'daily_anomalies': int(daily['is_order_anomaly'].sum()),
}
(OUT_DIR / 'cohort_time_series_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Saved to:', OUT_DIR)
print(summary)


## Next Notebook

Proceed to `06_deployment_scoring.ipynb`.
